# Exemple de notebook - filtres sql et récupération des données en pandas

L'objectif de ce notebook est de fournir des exemples pour pré-filtrer les données via sql avant de charger les données dans un DataFrame pandas.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)  # show all cols
pd.set_option("display.max_colwidth", None)  # show full width of showing cols
pd.set_option(
    "display.expand_frame_repr", False
)  # print cols side by side as it's supposed to be

In [2]:
# Nous commencons par importer les librairies nécessaires pour l'analyse des données.

import duckdb

ODIS_DUCKDB_FILE = "odis.duckdb"
PCC_DUCKDB_FILE = "dev.duckdb"

con = duckdb.connect(database=PCC_DUCKDB_FILE, read_only=True)
con.sql(f"ATTACH '{ODIS_DUCKDB_FILE}' AS odis;")

## Filtres

1. Filtrer les cat nat publiées depuis 2000


In [3]:
query_2020 = """
SELECT
	*
FROM dev.main.catnat_gaspar
WHERE dat_pub_arrete >= '2000-01-01'
ORDER BY dat_pub_arrete DESC
"""

cat_nat_2000 = con.sql(query_2020)
cat_nat_2000_df = cat_nat_2000.df()
cat_nat_2000_df.head(2)

,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin,dat_pub_arrete,dat_pub_jo,dat_maj
0,INTE2527885A,35066,Chartres-de-Bretagne,SEC,Sécheresse,2025-01-26 01:00:00,2025-01-26 01:00:00,2025-10-15 02:00:00,2025-10-23 02:00:00,2025-10-28 01:00:00
1,INTE2527885A,84014,Beaumont-de-Pertuis,SEC,Sécheresse,2023-03-31 02:00:00,2023-06-29 02:00:00,2025-10-15 02:00:00,2025-10-23 02:00:00,2025-10-28 01:00:00


2. Filtrer mes donées pour en récupérer une partie


In [4]:
where_clause = """
logements."YEAR" == '2022'
"""
query_2022 = f"""
SELECT
  *
FROM odis.main."gold_gold_logements_territoires" logements
WHERE
  {where_clause}
"""

logements_2022 = con.sql(query_2022)
logements_2022_df = logements_2022.df()
logements_2022_df.head(2)

,codgeo,YEAR,LOG,RP,RSECOCC,LOGVAC,MAISON,APPART,RPMAISON,RPAPPART,NB_MOY_PIECE,MEN,NBPI_RP
0,01001,2022.0,379.0,77.0,11.0,14.0,369.0,9.0,345.0,9.0,23.19481,77.0,1786.0
1,01002,2022.0,175.0,63.0,41.0,13.0,173.0,2.0,119.0,2.0,9.77778,63.0,616.0


## Selectionner des colonnes avant d'exécuter la requête


Selectionner les colonnes avant de charger les données permets une exécution plus rapide et limite l'usage de la mémoire.


In [5]:
query_rp = f"""
SELECT
  logements.codgeo,
  logements.RP as nombre_de_residences_principales
FROM odis.main."gold_gold_logements_territoires" logements
WHERE
  {where_clause}
"""
residences_principales_2022 = con.sql(query_rp)
residences_principales_2022_df = residences_principales_2022.df()
residences_principales_2022_df.head(2)

,codgeo,nombre_de_residences_principales
0,01001,77.0
1,01002,63.0


## Jointure

Joindre gold_gold_logements_territoires et gold_gold_emploi_bmo_secteurs pour croiser les données sur chaque commune :


In [6]:
query = f"""
SELECT
  logements.codgeo,
  logements.RP as nombre_de_residences_principales,
  emploi."Commerce" as nombre_d_emplois_dans_le_commerce
FROM
  odis.main."gold_gold_logements_territoires" logements
LEFT JOIN
  odis.main."gold_gold_emploi_bmo_secteurs" emploi
    ON logements.codgeo = emploi.codgeo
WHERE
  {where_clause}
"""


joined = con.sql(query)
joined_df = joined.df()
joined_df

,codgeo,nombre_de_residences_principales,nombre_d_emplois_dans_le_commerce
0,01,98924.53502,1613.0
1,01001,77.00000,191.0
2,01002,63.00000,418.0
3,01004,7107.01248,418.0
4,01005,90.30740,191.0
...,...,...,...
34961,reg75,910988.57759,28227.0
34962,reg76,785042.77899,24679.0
34963,reg84,61037.41312,33127.0
34964,reg93,686197.95107,23721.0


## Groupby et aggregats

Nombre total de prélèvements non conforme par commune en 2024


In [7]:
query = f"""
SELECT
  logements.codgeo,
  logements.RP as nombre_de_residences_principales,
  count(*) as nombre_de_cat_nat,
  coalesce(SUM(catnat.lib_risque_jo = 'Sécheresse'), 0) as nombre_de_cat_nat_secheresse
FROM
  odis.main."gold_gold_logements_territoires" logements
LEFT JOIN
  dev.main.catnat_gaspar catnat
    ON logements.codgeo = catnat.cod_commune
WHERE
  {where_clause}
GROUP BY
  logements.codgeo,
  logements.RP
"""
grouped = con.sql(query)
grouped_df = grouped.df()
grouped_df.sort_values("nombre_de_cat_nat", ascending=False)

,codgeo,nombre_de_residences_principales,nombre_de_cat_nat,nombre_de_cat_nat_secheresse
33866,06088,82437.10871,140,12.0
5718,49261,787.06742,103,19.0
13530,13055,9164.41734,98,25.0
20835,06027,753.84947,89,5.0
1349,49125,985.07642,83,22.0
...,...,...,...,...
34931,92,746653.12634,1,0.0
34959,83147,6.22222,1,0.0
34958,21435,64.00000,1,0.0
34957,60,7969.27510,1,0.0


: 